# Production, Distribution and Inventory Optimization

This project studies a four-period planning problem for a steel producer supplying eight customer locations from three factories. Production decisions, cutting patterns, customer assignments and vehicle routes are optimized together. The inventory model then allows earlier deliveries to be stored at customer warehouses.

The final section explores how the amount of available warehouse space affects distribution costs. All costs reported below come from an earlier set of solver runs; the full solution vectors and final solver logs were not retained, so the historical findings should not be read as newly reproduced results.

**Stack:** Python · Pyomo · Gurobi

By default, the notebook defines the models without launching the long solver runs. To execute an experiment, set its corresponding flag in the configuration cell to `True`.

## Run configuration

Model definitions and historical findings are available without a solver run. Solves are optional and may take substantial time.

In [ ]:
RUN_BASELINE = False
RUN_INVENTORY = False
RUN_SENSITIVITY = False

## Input data

The instance covers three factories, eight customers, three rebar sizes and four periods. Production capacity, customer demand, delivery distances and warehouse capacities are defined below. Bar weights use a steel density of 7.85 tonnes per cubic metre. Feasible cutting patterns are generated for 9 m and 12 m stock bars.

In [ ]:
import math
import time
import json
import pyomo.environ as pe
import pyomo.opt as po

F = ["IJmuiden", "Segal", "SouthWales"]
C = ["Bochum", "Boenen", "Dortmund", "Gelsenkirchen", "Hagen", "Iserlohn", "Neuss", "Schwerte"]
R = ["A", "B", "C"]
T = [1, 2, 3, 4]
T0 = [0] + T

Cap = {"IJmuiden": 12, "Segal": 10, "SouthWales": 28}
FSC = {"IJmuiden": 130, "Segal": 150, "SouthWales": 100}
VC = 3.0
Vcap = 10.0

BaseInvCap = {
    "Bochum": 10,
    "Boenen": 7,
    "Dortmund": 12,
    "Gelsenkirchen": 10,
    "Hagen": 12,
    "Iserlohn": 9,
    "Neuss": 8,
    "Schwerte": 5,
}

A_tbl = {
    1: {"Bochum": 2, "Boenen": 4, "Dortmund": 2, "Gelsenkirchen": 5, "Hagen": 19, "Iserlohn": 13, "Neuss": 20, "Schwerte": 4},
    2: {"Bochum": 6, "Boenen": 8, "Dortmund": 7, "Gelsenkirchen": 5, "Hagen": 23, "Iserlohn": 19, "Neuss": 16, "Schwerte": 5},
    3: {"Bochum": 5, "Boenen": 5, "Dortmund": 6, "Gelsenkirchen": 5, "Hagen": 25, "Iserlohn": 17, "Neuss": 14, "Schwerte": 3},
    4: {"Bochum": 3, "Boenen": 10, "Dortmund": 5, "Gelsenkirchen": 5, "Hagen": 16, "Iserlohn": 14, "Neuss": 26, "Schwerte": 4},
}
B_tbl = {
    1: {"Bochum": 4, "Boenen": 5, "Dortmund": 4, "Gelsenkirchen": 9, "Hagen": 15, "Iserlohn": 22, "Neuss": 12, "Schwerte": 2},
    2: {"Bochum": 5, "Boenen": 8, "Dortmund": 5, "Gelsenkirchen": 10, "Hagen": 33, "Iserlohn": 26, "Neuss": 23, "Schwerte": 8},
    3: {"Bochum": 7, "Boenen": 12, "Dortmund": 8, "Gelsenkirchen": 6, "Hagen": 31, "Iserlohn": 20, "Neuss": 30, "Schwerte": 2},
    4: {"Bochum": 8, "Boenen": 13, "Dortmund": 10, "Gelsenkirchen": 6, "Hagen": 33, "Iserlohn": 27, "Neuss": 30, "Schwerte": 6},
}
C_tbl = {
    1: {"Bochum": 6, "Boenen": 6, "Dortmund": 7, "Gelsenkirchen": 10, "Hagen": 12, "Iserlohn": 14, "Neuss": 22, "Schwerte": 5},
    2: {"Bochum": 7, "Boenen": 10, "Dortmund": 6, "Gelsenkirchen": 9, "Hagen": 35, "Iserlohn": 25, "Neuss": 32, "Schwerte": 6},
    3: {"Bochum": 7, "Boenen": 15, "Dortmund": 4, "Gelsenkirchen": 9, "Hagen": 33, "Iserlohn": 23, "Neuss": 31, "Schwerte": 7},
    4: {"Bochum": 7, "Boenen": 12, "Dortmund": 12, "Gelsenkirchen": 10, "Hagen": 38, "Iserlohn": 24, "Neuss": 31, "Schwerte": 2},
}

D = {}
for t in T:
    for c in C:
        D[(c, t, "A")] = A_tbl[t][c]
        D[(c, t, "B")] = B_tbl[t][c]
        D[(c, t, "C")] = C_tbl[t][c]

V_all = F + C
dist_matrix = {
    "IJmuiden":     {"IJmuiden":0, "Segal":284, "SouthWales":826, "Bochum":250, "Boenen":282, "Dortmund":266, "Gelsenkirchen":234, "Hagen":289, "Iserlohn":299, "Neuss":259, "Schwerte":279},
    "Segal":        {"IJmuiden":284, "Segal":0, "SouthWales":750, "Bochum":203, "Boenen":242, "Dortmund":222, "Gelsenkirchen":198, "Hagen":206, "Iserlohn":226, "Neuss":140, "Schwerte":216},
    "SouthWales":   {"IJmuiden":826, "Segal":750, "SouthWales":0, "Bochum":866, "Boenen":914, "Dortmund":885, "Gelsenkirchen":859, "Hagen":903, "Iserlohn":913, "Neuss":843, "Schwerte":901},
    "Bochum":       {"IJmuiden":250, "Segal":203, "SouthWales":866, "Bochum":0, "Boenen":55, "Dortmund":21, "Gelsenkirchen":19, "Hagen":41, "Iserlohn":51, "Neuss":56, "Schwerte":39},
    "Boenen":       {"IJmuiden":282, "Segal":242, "SouthWales":914, "Bochum":55, "Boenen":0, "Dortmund":34, "Gelsenkirchen":56, "Hagen":44, "Iserlohn":54, "Neuss":102, "Schwerte":32},
    "Dortmund":     {"IJmuiden":266, "Segal":222, "SouthWales":885, "Bochum":21, "Boenen":34, "Dortmund":0, "Gelsenkirchen":34, "Hagen":21, "Iserlohn":36, "Neuss":75, "Schwerte":15},
    "Gelsenkirchen":{"IJmuiden":234, "Segal":198, "SouthWales":859, "Bochum":19, "Boenen":56, "Dortmund":34, "Gelsenkirchen":0, "Hagen":56, "Iserlohn":66, "Neuss":62, "Schwerte":54},
    "Hagen":        {"IJmuiden":289, "Segal":206, "SouthWales":903, "Bochum":41, "Boenen":44, "Dortmund":21, "Gelsenkirchen":56, "Hagen":0, "Iserlohn":20, "Neuss":66, "Schwerte":14},
    "Iserlohn":     {"IJmuiden":299, "Segal":226, "SouthWales":913, "Bochum":51, "Boenen":54, "Dortmund":36, "Gelsenkirchen":66, "Hagen":20, "Iserlohn":0, "Neuss":85, "Schwerte":15},
    "Neuss":        {"IJmuiden":259, "Segal":140, "SouthWales":843, "Bochum":56, "Boenen":102, "Dortmund":75, "Gelsenkirchen":62, "Hagen":66, "Iserlohn":85, "Neuss":0, "Schwerte":75},
    "Schwerte":     {"IJmuiden":279, "Segal":216, "SouthWales":901, "Bochum":39, "Boenen":32, "Dortmund":15, "Gelsenkirchen":54, "Hagen":14, "Iserlohn":15, "Neuss":75, "Schwerte":0},
}
Dist = {(i, j): dist_matrix[i][j] for i in V_all for j in V_all}

rebar_len = {"A": 2.40, "B": 3.60, "C": 4.20}
diam = 0.057
radius = diam / 2
area = math.pi * radius * radius
density = 7.85
w_per_m = area * density
Wr = {r: rebar_len[r] * w_per_m for r in R}

def generate_patterns(L):
    LA, LB, LC = rebar_len["A"], rebar_len["B"], rebar_len["C"]
    pats = {}
    maxA, maxB, maxC = int(L // LA), int(L // LB), int(L // LC)

    for a in range(maxA + 1):
        for b in range(maxB + 1):
            for c in range(maxC + 1):
                used = a * LA + b * LB + c * LC
                if 1e-9 < used <= L + 1e-9:
                    pname = f"LB{int(L)}_A{a}_B{b}_C{c}"
                    pats[pname] = {"A": a, "B": b, "C": c, "L": L}
    return pats

PAT = {}
PAT.update(generate_patterns(9.0))
PAT.update(generate_patterns(12.0))
P = list(PAT.keys())
Arp = {(r, p): PAT[p][r] for r in R for p in P}
WLP = {p: PAT[p]["L"] * w_per_m for p in P}

## 1. Baseline production and routing

The baseline model meets each period's demand with deliveries made in the same period. It chooses cutting patterns and factory-to-customer shipments, subject to production and vehicle capacities. Each customer is served by at most one factory per period, and each active factory follows a single tour that starts and ends at its depot.

The objective combines fixed dispatch costs with distance-based transportation costs. Shipment quantities are modeled explicitly so that production, customer assignments and routes can be inspected separately.
**Run status:** The baseline model has not been re-solved since this code cleanup. The previously recorded objective was €29,047 with an optimal termination status; rerun locally to reproduce it.


In [ ]:
def solve_baseline():
    m = pe.ConcreteModel()

    m.F = pe.Set(initialize=F)
    m.C = pe.Set(initialize=C)
    m.R = pe.Set(initialize=R)
    m.T = pe.Set(initialize=T)
    m.P = pe.Set(initialize=P)

    def Arc_init(mm):
        idx = []
        for f in F:
            nodes = [f] + C
            for i in nodes:
                for j in nodes:
                    if i != j:
                        idx.append((f, i, j))
        return idx
    m.Arc = pe.Set(initialize=Arc_init, dimen=3)

    # X[f,p,t] = number of times cutting pattern p is used at factory f in period t
    m.X = pe.Var(m.F, m.P, m.T, domain=pe.NonNegativeIntegers)

    # Y[f,c,t,r] = units of rebar type r shipped from factory f to customer c in period t
    m.Y = pe.Var(m.F, m.C, m.T, m.R, domain=pe.NonNegativeIntegers)

    # S[f,c,t] = 1 if customer c is served by factory f in period t
    m.S = pe.Var(m.F, m.C, m.T, domain=pe.Binary)

    # U[f,t] = 1 if factory f is active in period t
    m.U = pe.Var(m.F, m.T, domain=pe.Binary)

    # Z[(f,i,j),t] = 1 if route arc i->j is used by factory f in period t
    m.Z = pe.Var(m.Arc, m.T, domain=pe.Binary)

    M = len(C)
    m.Pos = pe.Var(m.F, m.T, m.C, domain=pe.NonNegativeReals, bounds=(0, M))

    def obj_rule(mm):
        fixed = sum(FSC[f] * mm.U[f, t] for f in mm.F for t in mm.T)
        travel = sum(VC * Dist[(i, j)] * mm.Z[(f, i, j), t] for (f, i, j) in mm.Arc for t in mm.T)
        return fixed + travel
    m.Obj = pe.Objective(rule=obj_rule, sense=pe.minimize)

    m.Demand = pe.ConstraintList()
    for c in C:
        for t in T:
            for r in R:
                m.Demand.add(sum(m.Y[f, c, t, r] for f in F) == D[(c, t, r)])

    m.ProdBal = pe.ConstraintList()
    for f in F:
        for t in T:
            for r in R:
                m.ProdBal.add(sum(Arp[(r, p)] * m.X[f, p, t] for p in P) == sum(m.Y[f, c, t, r] for c in C))

    m.ProdCap = pe.ConstraintList()
    for f in F:
        for t in T:
            m.ProdCap.add(sum(WLP[p] * m.X[f, p, t] for p in P) <= Cap[f])

    m.VehCap = pe.ConstraintList()
    for f in F:
        for t in T:
            m.VehCap.add(sum(Wr[r] * m.Y[f, c, t, r] for c in C for r in R) <= Vcap * m.U[f, t])

    m.LinkYS = pe.ConstraintList()
    for f in F:
        for c in C:
            for t in T:
                for r in R:
                    m.LinkYS.add(m.Y[f, c, t, r] <= D[(c, t, r)] * m.S[f, c, t])

    m.AtMostOne = pe.ConstraintList()
    for c in C:
        for t in T:
            m.AtMostOne.add(sum(m.S[f, c, t] for f in F) <= 1)

    m.ActiveLink = pe.ConstraintList()
    for f in F:
        for c in C:
            for t in T:
                m.ActiveLink.add(m.S[f, c, t] <= m.U[f, t])

    m.DepotOut = pe.ConstraintList()
    m.DepotIn = pe.ConstraintList()
    for f in F:
        for t in T:
            m.DepotOut.add(sum(m.Z[(f, f, j), t] for j in C) == m.U[f, t])
            m.DepotIn.add(sum(m.Z[(f, i, f), t] for i in C) == m.U[f, t])

    m.CustIn = pe.ConstraintList()
    m.CustOut = pe.ConstraintList()
    for f in F:
        for t in T:
            for c in C:
                m.CustIn.add(sum(m.Z[(f, i, c), t] for i in ([f] + C) if i != c) == m.S[f, c, t])
                m.CustOut.add(sum(m.Z[(f, c, j), t] for j in ([f] + C) if j != c) == m.S[f, c, t])

    m.PosBounds = pe.ConstraintList()
    for f in F:
        for t in T:
            for c in C:
                m.PosBounds.add(m.Pos[f, t, c] >= m.S[f, c, t])
                m.PosBounds.add(m.Pos[f, t, c] <= M * m.S[f, c, t])

    m.MTZ = pe.ConstraintList()
    for f in F:
        for t in T:
            for i in C:
                for j in C:
                    if i != j:
                        m.MTZ.add(m.Pos[f, t, i] - m.Pos[f, t, j] + M * m.Z[(f, i, j), t] <= M - 1)

    solver = po.SolverFactory("gurobi")
    res = solver.solve(m, tee=False)
    return m, res

def print_baseline_results(m, res):
    print("\n" + "=" * 70)
    print("BASELINE RESULTS")
    print("=" * 70)
    print("Status:", res.solver.status)
    print("Termination:", res.solver.termination_condition)
    # Only report a solution if Gurobi has populated all model variables.
    required = list(m.component_data_objects(pe.Var, active=True))
    if any(pe.value(var, exception=False) is None for var in required):
        print("No complete solution is available to report.")
        return

    termination = res.solver.termination_condition
    if termination != po.TerminationCondition.optimal:
        print("Note: the solution is not proven optimal; check the solver log and bounds.")

    print("Objective value:", pe.value(m.Obj))

    def route_for(f, t):
        succ = {}
        for (ff, i, j) in m.Arc:
            if ff == f and pe.value(m.Z[(ff, i, j), t]) > 0.5:
                succ[i] = j
        if f not in succ:
            return [f]
        route = [f]
        cur = f
        for _ in range(len(C) + 5):
            cur = succ.get(cur, None)
            if cur is None:
                break
            route.append(cur)
            if cur == f:
                break
        return route

    for t in T:
        print(f"\n=== Period {t} ===")
        for f in F:
            if pe.value(m.U[f, t]) > 0.5:
                shipped = sum(Wr[r] * pe.value(m.Y[f, c, t, r]) for c in C for r in R)
                served = [c for c in C if pe.value(m.S[f, c, t]) > 0.5]
                print(f"Factory {f} active, shipped tonnes: {shipped:.4f}")
                print("  serves:", served)
                print("  route :", " -> ".join(route_for(f, t)))

# Solve the baseline model
if RUN_BASELINE:
    baseline_model, baseline_result = solve_baseline()
    print_baseline_results(baseline_model, baseline_result)

## 2. Production and routing with customer inventory

The second model allows finished rebars to be delivered ahead of demand and stored at customer warehouses. Inventory evolves according to

\[
I_{c,t,r} = I_{c,t-1,r} + \sum_f Y_{f,c,t,r} - D_{c,t,r}, \qquad I_{c,0,r}=0.
\]

The direct-delivery demand equality is replaced by this balance equation. Inventory must remain nonnegative and fit within each warehouse's weight capacity. Production and routing constraints continue to apply.

The solver uses a time limit and an optimality-gap target. When it stops before proving optimality, the reported cost is the best feasible solution found, not a certified optimum.

In [ ]:
def solve_with_inventory(time_limit=5400):
    Mship = {}
    for c in C:
        for t in T:
            for r in R:
                max_store = int(BaseInvCap[c] // Wr[r])
                Mship[(c, t, r)] = D[(c, t, r)] + max_store

    m = pe.ConcreteModel()
    m.F = pe.Set(initialize=F)
    m.C = pe.Set(initialize=C)
    m.R = pe.Set(initialize=R)
    m.T = pe.Set(initialize=T)
    m.T0 = pe.Set(initialize=T0)
    m.P = pe.Set(initialize=P)

    def Arc_init(mm):
        idx = []
        for f in F:
            nodes = [f] + C
            for i in nodes:
                for j in nodes:
                    if i != j:
                        idx.append((f, i, j))
        return idx
    m.Arc = pe.Set(initialize=Arc_init, dimen=3)

    m.X = pe.Var(m.F, m.P, m.T, domain=pe.NonNegativeIntegers)
    m.Y = pe.Var(m.F, m.C, m.T, m.R, domain=pe.NonNegativeIntegers)

    # I[c,t,r] = inventory of rebar type r at customer c at end of period t
    m.I = pe.Var(m.C, m.T0, m.R, domain=pe.NonNegativeIntegers)

    m.S = pe.Var(m.F, m.C, m.T, domain=pe.Binary)
    m.U = pe.Var(m.F, m.T, domain=pe.Binary)
    m.Z = pe.Var(m.Arc, m.T, domain=pe.Binary)

    Q = len(C)
    m.Pos = pe.Var(m.F, m.T, m.C, domain=pe.NonNegativeReals, bounds=(0, Q))

    def obj_rule(mm):
        fixed = sum(FSC[f] * mm.U[f, t] for f in mm.F for t in mm.T)
        travel = sum(VC * Dist[(i, j)] * mm.Z[(f, i, j), t] for (f, i, j) in mm.Arc for t in mm.T)
        return fixed + travel
    m.Obj = pe.Objective(rule=obj_rule, sense=pe.minimize)

    m.InvInit = pe.ConstraintList()
    for c in C:
        for r in R:
            m.InvInit.add(m.I[c, 0, r] == 0)

    m.InvBal = pe.ConstraintList()
    for c in C:
        for r in R:
            for t in T:
                m.InvBal.add(m.I[c, t, r] == m.I[c, t - 1, r] + sum(m.Y[f, c, t, r] for f in F) - D[(c, t, r)])

    m.WareCap = pe.ConstraintList()
    for c in C:
        for t in T:
            m.WareCap.add(sum(Wr[r] * m.I[c, t, r] for r in R) <= BaseInvCap[c])

    m.ProdBal = pe.ConstraintList()
    for f in F:
        for t in T:
            for r in R:
                m.ProdBal.add(sum(Arp[(r, p)] * m.X[f, p, t] for p in P) == sum(m.Y[f, c, t, r] for c in C))

    m.ProdCap = pe.ConstraintList()
    for f in F:
        for t in T:
            m.ProdCap.add(sum(WLP[p] * m.X[f, p, t] for p in P) <= Cap[f])

    m.VehCap = pe.ConstraintList()
    for f in F:
        for t in T:
            m.VehCap.add(sum(Wr[r] * m.Y[f, c, t, r] for c in C for r in R) <= Vcap * m.U[f, t])

    m.AtMostOne = pe.ConstraintList()
    for c in C:
        for t in T:
            m.AtMostOne.add(sum(m.S[f, c, t] for f in F) <= 1)

    m.ActiveLink = pe.ConstraintList()
    for f in F:
        for c in C:
            for t in T:
                m.ActiveLink.add(m.S[f, c, t] <= m.U[f, t])

    m.LinkYS = pe.ConstraintList()
    for f in F:
        for c in C:
            for t in T:
                for r in R:
                    m.LinkYS.add(m.Y[f, c, t, r] <= Mship[(c, t, r)] * m.S[f, c, t])

    m.DepotOut = pe.ConstraintList()
    m.DepotIn = pe.ConstraintList()
    for f in F:
        for t in T:
            m.DepotOut.add(sum(m.Z[(f, f, j), t] for j in C) == m.U[f, t])
            m.DepotIn.add(sum(m.Z[(f, i, f), t] for i in C) == m.U[f, t])

    m.CustIn = pe.ConstraintList()
    m.CustOut = pe.ConstraintList()
    for f in F:
        for t in T:
            nodes = [f] + C
            for c in C:
                m.CustIn.add(sum(m.Z[(f, i, c), t] for i in nodes if i != c) == m.S[f, c, t])
                m.CustOut.add(sum(m.Z[(f, c, j), t] for j in nodes if j != c) == m.S[f, c, t])

    m.PosBounds = pe.ConstraintList()
    for f in F:
        for t in T:
            for c in C:
                m.PosBounds.add(m.Pos[f, t, c] >= m.S[f, c, t])
                m.PosBounds.add(m.Pos[f, t, c] <= Q * m.S[f, c, t])

    m.MTZ = pe.ConstraintList()
    for f in F:
        for t in T:
            for i in C:
                for j in C:
                    if i != j:
                        m.MTZ.add(m.Pos[f, t, i] + 1 <= m.Pos[f, t, j] + Q * (1 - m.Z[(f, i, j), t]))

    solver = po.SolverFactory("gurobi")
    solver.options["TimeLimit"] = time_limit
    solver.options["MIPGap"] = 0.02
    solver.options["Threads"] = 0

    res = solver.solve(m, tee=True)
    return m, res



### Checking and saving an inventory solution

After Gurobi finishes, the next cells check every active Pyomo constraint, variable bound and integer restriction. A complete solution that passes is saved as `inventory_solution.json`, along with the solver status and objective value. The JSON file lets us inspect detailed shipments, cutting patterns, routes and inventories without repeating the solve.

The checker tests feasibility **for the implemented model**; it does not prove that the mathematical assumptions are complete or that a time-limited solution is optimal. The saved report figures are not used as substitute decision-variable values.

In [ ]:
from pathlib import Path


def check_solution(model, tolerance=1e-5):
    """Check variable bounds, integrality and every active model constraint."""
    missing = []
    violations = []
    checked = 0

    for var in model.component_data_objects(pe.Var, active=True):
        value = pe.value(var, exception=False)
        if value is None:
            missing.append(var.name)
            continue
        if var.lb is not None and value < pe.value(var.lb) - tolerance:
            violations.append((var.name, "lower bound", float(pe.value(var.lb) - value)))
        if var.ub is not None and value > pe.value(var.ub) + tolerance:
            violations.append((var.name, "upper bound", float(value - pe.value(var.ub))))
        if var.is_integer() and abs(value - round(value)) > tolerance:
            violations.append((var.name, "integrality", float(abs(value - round(value)))))

    if missing:
        return {"passed": False, "checked_constraints": 0,
                "missing_variables": missing[:20], "violation_count": None,
                "violations": []}

    for con in model.component_data_objects(pe.Constraint, active=True):
        checked += 1
        value = pe.value(con.body, exception=False)
        if value is None:
            violations.append((con.name, "unevaluable", None))
            continue
        if con.has_lb():
            residual = pe.value(con.lower) - value
            if residual > tolerance:
                violations.append((con.name, "lower bound", float(residual)))
        if con.has_ub():
            residual = value - pe.value(con.upper)
            if residual > tolerance:
                violations.append((con.name, "upper bound", float(residual)))

    return {"passed": not violations, "checked_constraints": checked,
            "missing_variables": [], "violation_count": len(violations),
            "violations": violations[:20]}


def export_inventory_solution(model, results, validation,
                              filename="inventory_solution.json"):
    """Save all decision-variable values and solver metadata for later checks."""
    if not validation["passed"]:
        raise ValueError("Solution validation failed. Nothing was exported.")

    solver = results.solver
    payload = {
        "model": "production_distribution_with_inventory",
        "solver_status": str(solver.status),
        "termination_condition": str(solver.termination_condition),
        "objective_eur": float(pe.value(model.Obj)),
        "validation": validation,
        "variables": {}
    }

    # Keep solver bounds when they were included in the results object.
    problem = next(iter(results.problem.values()), None)
    if problem is not None:
        for attr in ("lower_bound", "upper_bound"):
            value = getattr(problem, attr, None)
            if value is not None:
                try:
                    candidate = float(value)
                    if math.isfinite(candidate):
                        payload[attr] = candidate
                except (TypeError, ValueError):
                    pass

    for component in model.component_objects(pe.Var, active=True):
        payload["variables"][component.name] = [
            {"index": list(idx) if isinstance(idx, tuple) else [idx],
             "value": float(pe.value(component[idx]))}
            for idx in component
        ]

    with Path(filename).open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)
    return filename


In [ ]:
if RUN_INVENTORY:
    # Run this cell when a new inventory solution is needed.
    # The default time limit is 5,400 seconds; change it for a shorter test.
    inventory_model, inventory_result = solve_with_inventory(time_limit=5400)
    print("Status:", inventory_result.solver.status)
    print("Termination:", inventory_result.solver.termination_condition)

    validation = check_solution(inventory_model)
    print("Constraints checked:", validation["checked_constraints"])
    print("Validation passed:", validation["passed"])
    if not validation["passed"]:
        print("Missing variables:", validation["missing_variables"])
        print("First violations:", validation["violations"])
    else:
        objective = pe.value(inventory_model.Obj)
        print(f"Total cost: EUR {objective:,.2f}")
        if inventory_result.solver.termination_condition != po.TerminationCondition.optimal:
            print("Feasible solution; optimality has not been proven.")
        saved = export_inventory_solution(inventory_model, inventory_result, validation)
        print("Solution exported to:", saved)


## 3. Warehouse-capacity sensitivity analysis

Warehouse capacities are scaled from zero to their reference values. Each setting is solved using the inventory model, and the results track transportation and dispatch cost, active factory-period deliveries, solver termination status and runtime.

The objective does not include inventory holding or warehouse rental costs, so the analysis measures the operational value of available storage rather than the full economic cost of adding it. A time-limited solution is interpreted as a feasible incumbent, not necessarily the minimum possible cost.

**Execution note:** This section runs multiple optimization models and can take considerably longer than the preceding sections.

In [ ]:
def run_model_with_capacity_factor(cap_factor, time_limit):
    CurrentInvCap = {c: BaseInvCap[c] * cap_factor for c in C}

    Mship = {}
    for c in C:
        for t in T:
            for r in R:
                max_store = int(CurrentInvCap[c] // Wr[r]) if Wr[r] > 0 else 0
                Mship[(c, t, r)] = D[(c, t, r)] + max_store

    m = pe.ConcreteModel()
    m.F = pe.Set(initialize=F)
    m.C = pe.Set(initialize=C)
    m.R = pe.Set(initialize=R)
    m.T = pe.Set(initialize=T)
    m.T0 = pe.Set(initialize=T0)
    m.P = pe.Set(initialize=P)

    def N_init(mm, f):
        return [f] + C
    m.N = pe.Set(m.F, initialize=N_init)

    def Arc_init(mm):
        idx = []
        for f in F:
            nodes = [f] + C
            for i in nodes:
                for j in nodes:
                    if i != j:
                        idx.append((f, i, j))
        return idx
    m.Arc = pe.Set(initialize=Arc_init, dimen=3)

    m.X = pe.Var(m.F, m.P, m.T, domain=pe.NonNegativeIntegers)
    m.Y = pe.Var(m.F, m.C, m.T, m.R, domain=pe.NonNegativeIntegers)
    m.I = pe.Var(m.C, m.T0, m.R, domain=pe.NonNegativeIntegers)
    m.S = pe.Var(m.F, m.C, m.T, domain=pe.Binary)
    m.U = pe.Var(m.F, m.T, domain=pe.Binary)
    m.Z = pe.Var(m.Arc, m.T, domain=pe.Binary)

    Q_val = len(C)
    m.Pos = pe.Var(m.F, m.T, m.C, domain=pe.NonNegativeReals, bounds=(0, Q_val))

    def obj_rule(mm):
        fixed = sum(FSC[f] * mm.U[f, t] for f in mm.F for t in mm.T)
        travel = sum(VC * Dist[(i, j)] * mm.Z[(f, i, j), t] for (f, i, j) in mm.Arc for t in mm.T)
        return fixed + travel
    m.Obj = pe.Objective(rule=obj_rule, sense=pe.minimize)

    m.InvInit = pe.ConstraintList()
    for c in C:
        for r in R:
            m.InvInit.add(m.I[c, 0, r] == 0)

    m.InvBal = pe.ConstraintList()
    for c in C:
        for r in R:
            for t in T:
                m.InvBal.add(m.I[c, t, r] == m.I[c, t - 1, r] + sum(m.Y[f, c, t, r] for f in F) - D[(c, t, r)])

    m.WareCap = pe.ConstraintList()
    for c in C:
        for t in T:
            m.WareCap.add(sum(Wr[r] * m.I[c, t, r] for r in R) <= CurrentInvCap[c])

    m.ProdBal = pe.ConstraintList()
    for f in F:
        for t in T:
            for r in R:
                m.ProdBal.add(sum(Arp[(r, p)] * m.X[f, p, t] for p in P) == sum(m.Y[f, c, t, r] for c in C))

    m.ProdCap = pe.ConstraintList()
    for f in F:
        for t in T:
            m.ProdCap.add(sum(WLP[p] * m.X[f, p, t] for p in P) <= Cap[f])

    m.VehCap = pe.ConstraintList()
    for f in F:
        for t in T:
            m.VehCap.add(sum(Wr[r] * m.Y[f, c, t, r] for c in C for r in R) <= Vcap * m.U[f, t])

    m.AtMostOne = pe.ConstraintList()
    for c in C:
        for t in T:
            m.AtMostOne.add(sum(m.S[f, c, t] for f in F) <= 1)

    m.ActiveLink = pe.ConstraintList()
    for f in F:
        for c in C:
            for t in T:
                m.ActiveLink.add(m.S[f, c, t] <= m.U[f, t])

    m.LinkYS = pe.ConstraintList()
    for f in F:
        for c in C:
            for t in T:
                for r in R:
                    m.LinkYS.add(m.Y[f, c, t, r] <= Mship[(c, t, r)] * m.S[f, c, t])

    m.DepotOut = pe.ConstraintList()
    m.DepotIn = pe.ConstraintList()
    for f in F:
        for t in T:
            m.DepotOut.add(sum(m.Z[(f, f, j), t] for j in C) == m.U[f, t])
            m.DepotIn.add(sum(m.Z[(f, i, f), t] for i in C) == m.U[f, t])

    m.CustIn = pe.ConstraintList()
    m.CustOut = pe.ConstraintList()
    for f in F:
        for t in T:
            nodes = [f] + C
            for c in C:
                m.CustIn.add(sum(m.Z[(f, i, c), t] for i in nodes if i != c) == m.S[f, c, t])
                m.CustOut.add(sum(m.Z[(f, c, j), t] for j in nodes if j != c) == m.S[f, c, t])

    m.PosBounds = pe.ConstraintList()
    for f in F:
        for t in T:
            for c in C:
                m.PosBounds.add(m.Pos[f, t, c] >= m.S[f, c, t])
                m.PosBounds.add(m.Pos[f, t, c] <= Q_val * m.S[f, c, t])

    m.MTZ = pe.ConstraintList()
    for f in F:
        for t in T:
            for i in C:
                for j in C:
                    if i != j:
                        m.MTZ.add(m.Pos[f, t, i] + 1 <= m.Pos[f, t, j] + Q_val * (1 - m.Z[(f, i, j), t]))

    solver = po.SolverFactory("gurobi")
    solver.options["TimeLimit"] = time_limit
    solver.options["MIPGap"] = 0.02
    solver.options["Threads"] = 0

    start_time = time.time()
    res = solver.solve(m, tee=False)
    end_time = time.time()

    solve_time = end_time - start_time
    try:
        obj_val = pe.value(m.Obj)
    except ValueError:
        obj_val = None

    active_trucks = 0
    if obj_val is not None:
        active_trucks = sum(pe.value(m.U[f, t]) for f in F for t in T if pe.value(m.U[f, t]) > 0.5)

    return {
        "Factor": cap_factor,
        "Objective": obj_val,
        "SolveTime_sec": round(solve_time, 2),
        "Status": str(res.solver.status),
        "Termination": str(res.solver.termination_condition),
        "ActiveTrucks": active_trucks
    }

def run_capacity_sensitivity():
    iteration_time_limit = 5400
    factors_to_test = [0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1, 0.2, 0.5, 1.0]
    results_list = []

    print("Start Sensitivity Analysis...")
    for factor in factors_to_test:
        results_list.append(run_model_with_capacity_factor(factor, time_limit=iteration_time_limit))

    return results_list

def print_sensitivity_results(results_list):
    print("\n" + "=" * 70)
    print("PART 3 RESULTS - SENSITIVITY ANALYSIS")
    print("=" * 70)
    print(f"{'Factor':<10} | {'Objective (EUR)':<18} | {'Active Trucks':<15} | {'Time (s)':<10} | {'Termination'}")
    print("-" * 90)
    for r in results_list:
        obj_str = f"{r['Objective']:.2f}" if r["Objective"] is not None else "Infeasible/No Sol"
        print(f"{r['Factor']:<10} | {obj_str:<18} | {r['ActiveTrucks']:<15} | {r['SolveTime_sec']:<10} | {r['Termination']}")

    with open("sensitivity_results.json", "w") as f:
        json.dump(results_list, f)

# Run warehouse-capacity scenarios
if RUN_SENSITIVITY:
    capacity_results = run_capacity_sensitivity()
    print_sensitivity_results(capacity_results)

## Results from the planning scenarios

| Scenario | Total cost | Status of the historical result |
|:--|--:|:--|
| Direct delivery | €29,047 | Reported as optimal; cost agrees with the published routes |
| Customer inventory | €18,134 | Feasible incumbent, not proven optimal; cost agrees with the published routes |

The inventory scenario's reported cost is about **37.6% lower** than the direct-delivery baseline. This comparison does not include inventory holding or warehouse rental costs. The earlier inventory run reportedly stopped at a 3.52% optimality gap; the final solver log and full decision-variable values are unavailable for independent confirmation.

In the warehouse-capacity experiments, selected historical results were €29,053 at zero capacity, €19,553 at 7% of nominal capacity, and €18,134 at full capacity. These are *reported outputs*, not a fully verified cost curve; intermediate points and solver gaps have not been reproduced. See `results/historical_results.csv`.

A small difference between the zero-inventory sensitivity result and the direct-delivery benchmark should not be attributed to the use of big-M constraints alone: if both models describe the same feasible region and reach proven optimality, a valid big-M reformulation should not change the optimum. This difference remains unresolved without the original solver details.

The main limitation is that the notebook does not contain the complete historical variable assignments. The validation and export functions are available for any future rerun, but they have not been used to certify the reported historical solution.